# Оброблення тексту

## Аналіз за допомогою UDPipe

In [1]:
import json

import requests

In [2]:
INPUT_FILE = "Zabrodin_file_1.txt"
OUTPUT_FILE = "Zabrodin_file_2.json"
URL = "http://localhost:3000/process"

In [3]:
def load_text(file_path: str) -> str:
    print(f"Reading {file_path}")
    with open(file_path, encoding="utf-8") as f:
        return f.read()

In [4]:
def save_json(data: dict, file_path: str):
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"Output written to {file_path}")

In [5]:
def analyze(text: str, url: str) -> dict:
    print(f"Sending {len(text)} characters to {URL}")

    try:
        response = requests.post(url, data={
            "data": text,
            "tokenizer": "",
            "tagger": "",
            "parser": "",
        })
        response.raise_for_status()
    except requests.exceptions.ConnectionError:
        print(f"Could not connect to {url}")
        raise

    return response.json()

In [6]:
text_data = load_text(INPUT_FILE)
json_data = analyze(text_data, URL)
save_json(json_data, OUTPUT_FILE)

Reading Zabrodin_file_1.txt
Sending 267951 characters to http://localhost:3000/process
Output written to Zabrodin_file_2.json


## Видобування токенів та їх POS-тегів

In [7]:
INPUT_FILE = "Zabrodin_file_2.json"
OUTPUT_FILE = "Zabrodin_file_3.txt"

In [8]:
def save_tokens(tokens: list[tuple[str, str]], file_path: str):
    with open(file_path, mode="w", encoding="utf-8") as f:
        for lemma, u_pos_tag in tokens:
            f.write(f"{lemma} {u_pos_tag}\n")

    print(f"{len(tokens)} tokens written to {file_path}")


def is_punctuation(pos_tag: str) -> bool:
    return pos_tag == "PUNCT"


def extract_tokens(json: str) -> list[tuple[str, str]]:
    tokens = []
    for line in json.splitlines():
        if not line or line.startswith("#"):
            continue

        fields: list[str] = line.split("\t")
        if len(fields) < 10:
            continue

        lemma, pos_tag = fields[2], fields[3]
        if is_punctuation(pos_tag):
            continue

        tokens.append((lemma, pos_tag))

    return tokens


tokens_data = extract_tokens(json_data["result"])
save_tokens(tokens_data, OUTPUT_FILE)

44398 tokens written to Zabrodin_file_3.txt


In [9]:
display(tokens_data[:10])

[('кайдашевий', 'ADJ'),
 ('сім’я', 'NOUN'),
 ('Іван', 'PROPN'),
 ('Нечуй', 'PROPN'),
 ('Левицький', 'PROPN'),
 ('повість', 'NOUN'),
 ('написати', 'VERB'),
 ('в', 'ADP'),
 ('1878', 'ADJ'),
 ('рік', 'NOUN')]